# Modelado en SQL: Emisiones de GEI de Argentina (1990–2022)

En el notebook anterior dejamos el inventario limpio. Acá le damos otra forma: lo llevamos a una base de datos SQLite, organizada en un **esquema en estrella**, y construimos sobre ella siete vistas que responden las preguntas del proyecto. Esas tablas y vistas son las que después se importan en Power BI para armar el dashboard.

El recorrido: preparamos los datos y la base (sección 1), diseñamos el modelo (2), cargamos y verificamos las tablas (3), construimos las vistas (4), las contrastamos contra los datos originales (5) y exportamos todo a CSV (6).

Los valores están en **MtCO₂e** (millones de toneladas de CO₂ equivalente), como interpretamos en el notebook 01. Las columnas conservan `tco2e` en el nombre (`valor_tco2e`, `total_tco2e`) porque el dashboard ya se apoya en ellas.

## 1. Preparación de los datos y de la base

### 1.1 Librerías y rutas

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# La raíz del proyecto se busca hacia arriba desde la carpeta de trabajo, así el notebook
# funciona se abra Jupyter donde se abra (raíz, notebook/ o una carpeta que contenga el proyecto)
def encontrar_raiz(archivo=("data", "raw", "emisiones_datos_totales_1990_2022.csv")):
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for candidata in (base, base / "emisiones-gei-argentina"):
            if candidata.joinpath(*archivo).exists():
                return candidata
    raise FileNotFoundError("No se encontró data/raw/" + archivo[-1] + " desde " + str(cwd))

ROOT = encontrar_raiz()
PROCESSED = ROOT / "data" / "processed"
INTERIM = ROOT / "data" / "interim"
SQL = ROOT / "sql"

for carpeta in (INTERIM, SQL):
    carpeta.mkdir(parents=True, exist_ok=True)

### 1.2 Lectura del dataset limpio

In [2]:
df = pd.read_csv(PROCESSED / "emisiones_gei_1990_2022.csv")
df = df.rename(columns={
    "año": "anio",
    "valor_en_toneladas_de_co2e": "valor_tco2e",
})
df.shape

(7737, 9)

Partimos del CSV con el que cerró el notebook 01: 7.737 registros y 9 columnas. Renombramos dos columnas para que sean más cómodas de escribir en SQL, sin ñ y sin nombres largos.

### 1.3 La base de datos

In [3]:
DB = SQL / "proyecto_gei.db"

conn = sqlite3.connect(DB)
cur = conn.cursor()

Usamos **SQLite** porque no necesita instalar ni levantar un servidor: guarda todo en un único archivo (`proyecto_gei.db`) y viene incluido en Python. Para un proyecto que se publica tal cual en GitHub, es la opción más simple.

## 2. Diseño del modelo: esquema en estrella

En lugar de dejar todo en una tabla plana como el CSV, organizamos los datos en un **esquema en estrella**, que es el modelo que mejor entienden las herramientas de BI como Power BI:

* **`fact_emisiones`** es la tabla de hechos y tiene una fila por cada registro del inventario: el año, la actividad, la subactividad, la categoría, el código IPCC, el valor y el tipo de registro, más dos claves que apuntan a las dimensiones.
* **`dim_sector`** y **`dim_gas`** son tablas chicas, tipo catálogo. En vez de repetir miles de veces el texto "Agricultura, Ganadería…", guardamos un número (`sector_id`) que apunta a esa tabla.

```
dim_sector ──< fact_emisiones >── dim_gas
```

Así se evita duplicar texto y cometer errores de tipeo entre filas, y es la misma estructura de relaciones que después se arma en Power BI.

In [4]:
cur.executescript("""
DROP TABLE IF EXISTS fact_emisiones;
DROP TABLE IF EXISTS dim_sector;
DROP TABLE IF EXISTS dim_gas;

CREATE TABLE dim_sector (
    sector_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    sector      TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_gas (
    gas_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    tipo_de_gas TEXT NOT NULL UNIQUE,
    nombre_gas  TEXT NOT NULL
);

CREATE TABLE fact_emisiones (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    anio                INTEGER NOT NULL,
    sector_id           INTEGER NOT NULL REFERENCES dim_sector(sector_id),
    actividad           TEXT NOT NULL,
    subactividad        TEXT NOT NULL,
    categoria           TEXT NOT NULL,
    id_ipcc             TEXT NOT NULL,
    gas_id              INTEGER NOT NULL REFERENCES dim_gas(gas_id),
    valor_tco2e         REAL NOT NULL,
    tipo_de_registro    TEXT NOT NULL
);
""")
conn.commit()

## 3. Carga y verificación de las tablas

Cargamos primero las dos dimensiones y después la tabla de hechos. Cada paso se verifica antes de seguir.

### 3.1 Dimensiones

In [5]:
sectores = sorted(df["sector"].unique())
cur.executemany("INSERT INTO dim_sector (sector) VALUES (?)", [(s,) for s in sectores])

nombres_gas = {
    "CO2": "Dióxido de carbono",
    "CH4": "Metano",
    "N2O": "Óxido nitroso",
    "HFC": "Hidrofluorocarbonos",
    "PFC": "Perfluorocarbonos",
}
gases = sorted(df["tipo_de_gas"].unique())
cur.executemany(
    "INSERT INTO dim_gas (tipo_de_gas, nombre_gas) VALUES (?, ?)",
    [(g, nombres_gas[g]) for g in gases],
)
conn.commit()

display(pd.read_sql("SELECT * FROM dim_sector", conn))
display(pd.read_sql("SELECT * FROM dim_gas", conn))

,sector_id,sector
0,1,"Agricultura, Ganadería, Silvicultura y Otros U..."
1,2,Energía
2,3,Procesos industriales y uso de productos
3,4,Residuos


,gas_id,tipo_de_gas,nombre_gas
0,1,CH4,Metano
1,2,CO2,Dióxido de carbono
2,3,HFC,Hidrofluorocarbonos
3,4,N2O,Óxido nitroso
4,5,PFC,Perfluorocarbonos


Cada sector y cada gas recibió su propio identificador autogenerado (`sector_id`, `gas_id`).

### 3.2 Tabla de hechos

In [6]:
sector_ids = pd.read_sql("SELECT sector_id, sector FROM dim_sector", conn)
gas_ids = pd.read_sql("SELECT gas_id, tipo_de_gas FROM dim_gas", conn)

# Reemplazamos el texto de sector y gas por los identificadores de las dimensiones
df = df.merge(sector_ids, on="sector", how="left")
df = df.merge(gas_ids, on="tipo_de_gas", how="left")

assert df["sector_id"].isnull().sum() == 0
assert df["gas_id"].isnull().sum() == 0

columnas_fact = ["anio", "sector_id", "actividad", "subactividad",
                 "categoria", "id_ipcc", "gas_id", "valor_tco2e", "tipo_de_registro"]

df[columnas_fact].to_sql("fact_emisiones", conn, if_exists="append", index=False)
conn.commit()

Antes de cargar, traemos los identificadores de las dimensiones y los usamos para reemplazar el texto de sector y gas. Los dos `assert` confirman que ninguna fila quedó sin su clave.

### 3.3 Control del total

In [7]:
total_filas = cur.execute("SELECT COUNT(*) FROM fact_emisiones").fetchone()[0]
total_valor = cur.execute("SELECT SUM(valor_tco2e) FROM fact_emisiones").fetchone()[0]

assert total_filas == len(df)
assert abs(total_valor - 12398.33) < 0.01

print(f"Filas: {total_filas}")
print(f"Valor total (MtCO₂e): {total_valor:,.2f}")

Filas: 7737
Valor total (MtCO₂e): 12,398.33


La tabla de hechos tiene las 7.737 filas del dataset limpio y suma 12.398,33 MtCO₂e, el mismo total que validamos en el notebook 01. No se perdió ni se duplicó ningún registro en el camino.

## 4. Vistas analíticas

Una **vista** es una consulta guardada que se comporta como una tabla pero no duplica datos: cada vez que se consulta, SQLite recalcula el resultado a partir de las tablas originales. Sirve para dejar resuelto del lado de la base todo lo que no depende de los filtros del dashboard.

Construimos siete vistas, una por pregunta del proyecto. Son agregaciones fijas (totales, porcentajes, rankings) que Power BI importa como tablas. Los cálculos que sí cambian según lo que filtre el usuario, como un total que responde al sector elegido, no van acá: se resuelven en Power BI con medidas DAX.

In [8]:
def ver(vista, filas=8):
    """Muestra la cantidad de filas de una vista y sus primeras filas."""
    tabla = pd.read_sql(f"SELECT * FROM {vista}", conn)
    print(f"{vista}: {len(tabla)} filas")
    return tabla.head(filas)

### 4.1 Balance anual: emisiones, remociones y balance neto

Responde la pregunta principal: ¿cómo evolucionó el balance año a año? Separamos las emisiones (valores positivos) de las remociones (valores negativos, asociadas a bosques y tierras que capturan carbono) para mostrarlas por separado en el dashboard, además del balance neto.

In [9]:
cur.executescript("""
DROP VIEW IF EXISTS vw_balance_anual;
CREATE VIEW vw_balance_anual AS
SELECT
    anio,
    ROUND(SUM(CASE WHEN valor_tco2e > 0 THEN valor_tco2e ELSE 0 END), 2) AS emisiones,
    ROUND(SUM(CASE WHEN valor_tco2e < 0 THEN valor_tco2e ELSE 0 END), 2) AS remociones,
    ROUND(SUM(valor_tco2e), 2) AS balance_neto
FROM fact_emisiones
GROUP BY anio
ORDER BY anio;
""")
conn.commit()

ver("vw_balance_anual")

vw_balance_anual: 33 filas


,anio,emisiones,remociones,balance_neto
0,1990,313.43,-39.71,273.72
1,1991,343.90,-39.72,304.18
2,1992,346.61,-34.57,312.04
3,1993,353.19,-43.68,309.51
4,1994,352.98,-37.15,315.83
5,1995,363.95,-42.23,321.72
6,1996,395.43,-40.05,355.38
7,1997,405.94,-53.57,352.37


### 4.2 Totales y participación por sector

Responde qué sectores explican la mayor parte del balance acumulado entre 1990 y 2022. Además del total, calculamos el porcentaje de cada sector sobre el total general, listo para un gráfico de barras o de torta.

In [10]:
cur.executescript("""
DROP VIEW IF EXISTS vw_totales_sector;
CREATE VIEW vw_totales_sector AS
SELECT
    s.sector,
    ROUND(SUM(f.valor_tco2e), 2) AS total_tco2e,
    ROUND(100.0 * SUM(f.valor_tco2e) / (SELECT SUM(valor_tco2e) FROM fact_emisiones), 1) AS participacion_pct
FROM fact_emisiones f
JOIN dim_sector s ON f.sector_id = s.sector_id
GROUP BY s.sector
ORDER BY total_tco2e DESC;
""")
conn.commit()

ver("vw_totales_sector")

vw_totales_sector: 4 filas


,sector,total_tco2e,participacion_pct
0,"Agricultura, Ganadería, Silvicultura y Otros U...",5899.92,47.6
1,Energía,5422.41,43.7
2,Residuos,576.18,4.6
3,Procesos industriales y uso de productos,499.82,4.0


### 4.3 Evolución por año y sector

Es la misma información que el balance anual, pero desagregada por sector y en **formato largo**: una fila por combinación de año y sector, en lugar de una columna por sector. Es el formato que necesita Power BI para dibujar un gráfico de líneas con una serie por sector.

In [11]:
cur.executescript("""
DROP VIEW IF EXISTS vw_evolucion_sector_anio;
CREATE VIEW vw_evolucion_sector_anio AS
SELECT
    f.anio,
    s.sector,
    ROUND(SUM(f.valor_tco2e), 2) AS total_tco2e
FROM fact_emisiones f
JOIN dim_sector s ON f.sector_id = s.sector_id
GROUP BY f.anio, s.sector
ORDER BY f.anio, s.sector;
""")
conn.commit()

ver("vw_evolucion_sector_anio")

vw_evolucion_sector_anio: 132 filas


,anio,sector,total_tco2e
0,1990,"Agricultura, Ganadería, Silvicultura y Otros U...",151.13
1,1990,Energía,102.30
2,1990,Procesos industriales y uso de productos,8.48
3,1990,Residuos,11.81
4,1991,"Agricultura, Ganadería, Silvicultura y Otros U...",176.17
5,1991,Energía,108.01
6,1991,Procesos industriales y uso de productos,7.87
7,1991,Residuos,12.13


### 4.4 Variación interanual por sector (función de ventana `LAG`)

Calcula cuánto cambió cada sector respecto del año anterior, en valor absoluto y en porcentaje. Para eso usamos `LAG()`, una **función de ventana** que permite mirar la fila del año anterior del mismo sector sin hacer un `JOIN` adicional. El primer año de cada sector no tiene año previo con el que compararse, así que su variación queda en `NULL`.

In [12]:
cur.executescript("""
DROP VIEW IF EXISTS vw_variacion_sector;
CREATE VIEW vw_variacion_sector AS
WITH totales AS (
    SELECT f.anio, s.sector, SUM(f.valor_tco2e) AS total_tco2e
    FROM fact_emisiones f
    JOIN dim_sector s ON f.sector_id = s.sector_id
    GROUP BY f.anio, s.sector
)
SELECT
    anio,
    sector,
    ROUND(total_tco2e, 2) AS total_tco2e,
    ROUND(total_tco2e - LAG(total_tco2e) OVER (PARTITION BY sector ORDER BY anio), 2) AS variacion_absoluta,
    ROUND(
        100.0 * (total_tco2e - LAG(total_tco2e) OVER (PARTITION BY sector ORDER BY anio))
        / NULLIF(ABS(LAG(total_tco2e) OVER (PARTITION BY sector ORDER BY anio)), 0)
    , 1) AS variacion_pct
FROM totales
ORDER BY sector, anio;
""")
conn.commit()

pd.read_sql("SELECT * FROM vw_variacion_sector WHERE anio IN (1990, 1991, 2022)", conn)

,anio,sector,total_tco2e,variacion_absoluta,variacion_pct
0,1990,"Agricultura, Ganadería, Silvicultura y Otros U...",151.13,NaN,NaN
1,1991,"Agricultura, Ganadería, Silvicultura y Otros U...",176.17,25.04,16.6
2,2022,"Agricultura, Ganadería, Silvicultura y Otros U...",153.55,-6.74,-4.2
3,1990,Energía,102.30,NaN,NaN
4,1991,Energía,108.01,5.71,5.6
5,2022,Energía,200.32,7.49,3.9
6,1990,Procesos industriales y uso de productos,8.48,NaN,NaN
7,1991,Procesos industriales y uso de productos,7.87,-0.61,-7.2
8,2022,Procesos industriales y uso de productos,23.59,1.48,6.7
9,1990,Residuos,11.81,NaN,NaN


### 4.5 Composición por tipo de gas y año (función de ventana `SUM() OVER`)

Responde qué gas explica la mayor parte de las emisiones en cada año y cómo cambió esa composición. `SUM() OVER` es otra función de ventana: calcula el total del año sin colapsar las filas, como lo haría un `GROUP BY`, y así podemos dividir cada gas por el total de su propio año dentro de la misma consulta.

In [13]:
cur.executescript("""
DROP VIEW IF EXISTS vw_composicion_gas_anio;
CREATE VIEW vw_composicion_gas_anio AS
WITH totales_gas AS (
    SELECT f.anio, g.tipo_de_gas, SUM(f.valor_tco2e) AS total_tco2e
    FROM fact_emisiones f
    JOIN dim_gas g ON f.gas_id = g.gas_id
    GROUP BY f.anio, g.tipo_de_gas
)
SELECT
    anio,
    tipo_de_gas,
    ROUND(total_tco2e, 2) AS total_tco2e,
    ROUND(100.0 * total_tco2e / SUM(total_tco2e) OVER (PARTITION BY anio), 1) AS participacion_pct
FROM totales_gas
ORDER BY anio, tipo_de_gas;
""")
conn.commit()

ver("vw_composicion_gas_anio")

vw_composicion_gas_anio: 158 filas


,anio,tipo_de_gas,total_tco2e,participacion_pct
0,1990,CH4,115.93,42.4
1,1990,CO2,141.67,51.8
2,1990,N2O,16.07,5.9
3,1990,PFC,0.05,0.0
4,1991,CH4,118.19,38.9
5,1991,CO2,169.38,55.7
6,1991,N2O,16.56,5.4
7,1991,PFC,0.05,0.0


### 4.6 Las 10 categorías con mayor balance en 2018-2022 (función de ventana `RANK`)

Identifica las diez categorías individuales, como "Bovinos de Carne" o "Transporte terrestre por carretera", que más aportaron al balance en los últimos cinco años del dataset. `RANK()` ordena las categorías de mayor a menor y nos quedamos con las primeras diez.

In [14]:
cur.executescript("""
DROP VIEW IF EXISTS vw_top_categorias_recientes;
CREATE VIEW vw_top_categorias_recientes AS
WITH totales_categoria AS (
    SELECT categoria, SUM(valor_tco2e) AS total_tco2e
    FROM fact_emisiones
    WHERE anio >= 2018
    GROUP BY categoria
),
rankeadas AS (
    SELECT categoria, total_tco2e, RANK() OVER (ORDER BY total_tco2e DESC) AS ranking
    FROM totales_categoria
)
SELECT ranking, categoria, ROUND(total_tco2e, 2) AS total_tco2e
FROM rankeadas
WHERE ranking <= 10
ORDER BY ranking;
""")
conn.commit()

ver("vw_top_categorias_recientes", filas=10)

vw_top_categorias_recientes: 10 filas


,ranking,categoria,total_tco2e
0,1,Bovinos de Carne,361.49
1,2,Transporte terrestre por carretera,225.97
2,3,Producción pública de electricidad y calor,196.20
3,4,Tierras forestales convertidas en pastizales,172.58
4,5,Residencial,121.01
5,6,Tierras forestales convertidas en Tierras de c...,109.02
6,7,Fugitivas producción de Gas Natural,100.72
7,8,Industria no especificada:,53.84
8,9,Fabricación de combustibles sólidos y otras in...,53.68
9,10,Bovinos Lecheros,45.32


### 4.7 Actividades dentro de Energía y Agricultura/Ganadería

Abre los dos sectores de mayor peso por actividad, para poder explicar en el dashboard qué actividades puntuales sostienen el total de cada uno.

In [15]:
cur.executescript("""
DROP VIEW IF EXISTS vw_actividades_sector_principal;
CREATE VIEW vw_actividades_sector_principal AS
SELECT
    s.sector,
    f.actividad,
    ROUND(SUM(f.valor_tco2e), 2) AS total_tco2e
FROM fact_emisiones f
JOIN dim_sector s ON f.sector_id = s.sector_id
WHERE s.sector IN ('Energía', 'Agricultura, Ganadería, Silvicultura y Otros Usos de la Tierra')
GROUP BY s.sector, f.actividad
ORDER BY s.sector, total_tco2e DESC;
""")
conn.commit()

ver("vw_actividades_sector_principal")

vw_actividades_sector_principal: 5 filas


,sector,actividad,total_tco2e
0,"Agricultura, Ganadería, Silvicultura y Otros U...",Ganadería,2701.09
1,"Agricultura, Ganadería, Silvicultura y Otros U...",Tierra,2368.58
2,"Agricultura, Ganadería, Silvicultura y Otros U...",Uso de Suelos,830.25
3,Energía,Actividades de quema de combustible,4669.41
4,Energía,Emisiones fugitivas provenientes de la fabrica...,753.00


## 5. Control de las vistas

Una vista puede estar bien escrita y dar un resultado equivocado sin que nada falle. Por eso recalculamos cada una con pandas, a partir del dataset limpio, y comparamos fila por fila. Los resultados de SQL están redondeados a dos decimales (uno en los porcentajes), así que aceptamos diferencias de hasta ese redondeo.

In [16]:
def leer(vista, claves):
    return pd.read_sql(f"SELECT * FROM {vista}", conn).set_index(claves).sort_index()


def verificar(vista, esperado, claves, tolerancia=0.011):
    sql = leer(vista, claves)
    esperado = esperado.sort_index()
    assert sql.index.equals(esperado.index), f"{vista}: las filas no coinciden"
    assert sql.columns.tolist() == esperado.columns.tolist(), f"{vista}: las columnas no coinciden"
    assert sql.isna().equals(esperado.isna()), f"{vista}: los nulos no coinciden"
    diferencia = (sql - esperado).abs().max().max()
    assert diferencia <= tolerancia, f"{vista}: diferencia máxima {diferencia}"
    print(f"OK  {vista}: {len(sql)} filas, diferencia máxima {diferencia:.4f}")


V = "valor_tco2e"
SECTORES_PRINCIPALES = ["Energía", "Agricultura, Ganadería, Silvicultura y Otros Usos de la Tierra"]

# 4.1 Balance anual
verificar("vw_balance_anual", pd.DataFrame({
    "emisiones": df[df[V] > 0].groupby("anio")[V].sum(),
    "remociones": df[df[V] < 0].groupby("anio")[V].sum(),
    "balance_neto": df.groupby("anio")[V].sum(),
}).round(2), ["anio"])

# 4.2 Totales por sector
tot_sector = df.groupby("sector")[V].sum()
verificar("vw_totales_sector", pd.DataFrame({
    "total_tco2e": tot_sector.round(2),
    "participacion_pct": (tot_sector / tot_sector.sum() * 100).round(1),
}), ["sector"], tolerancia=0.11)

# 4.3 Evolución por año y sector
evo = df.groupby(["anio", "sector"])[V].sum()
verificar("vw_evolucion_sector_anio", pd.DataFrame({"total_tco2e": evo.round(2)}), ["anio", "sector"])

# 4.4 Variación interanual
evo_ord = evo.sort_index(level=["sector", "anio"])
previo = evo_ord.groupby(level="sector").shift(1)
variacion = pd.DataFrame({
    "total_tco2e": evo.round(2),
    "variacion_absoluta": (evo - previo.reindex(evo.index)).round(2),
    "variacion_pct": ((evo - previo.reindex(evo.index)) / previo.reindex(evo.index).abs() * 100).round(1),
})
verificar("vw_variacion_sector", variacion, ["anio", "sector"], tolerancia=0.11)

# 4.5 Composición por gas
gas = df.groupby(["anio", "tipo_de_gas"])[V].sum()
verificar("vw_composicion_gas_anio", pd.DataFrame({
    "total_tco2e": gas.round(2),
    "participacion_pct": (gas / gas.groupby(level="anio").transform("sum") * 100).round(1),
}), ["anio", "tipo_de_gas"], tolerancia=0.11)

# 4.6 Top 10 de categorías, 2018-2022
top = df[df["anio"] >= 2018].groupby("categoria")[V].sum().sort_values(ascending=False).head(10)
esperado_top = pd.DataFrame({"categoria": top.index, "total_tco2e": top.values.round(2)},
                            index=pd.Index(range(1, 11), name="ranking"))
sql_top = leer("vw_top_categorias_recientes", ["ranking"])
assert sql_top["categoria"].tolist() == esperado_top["categoria"].tolist(), "vw_top_categorias_recientes: el orden no coincide"
dif = (sql_top["total_tco2e"] - esperado_top["total_tco2e"]).abs().max()
assert dif <= 0.011
print(f"OK  vw_top_categorias_recientes: 10 filas, diferencia máxima {dif:.4f}")

# 4.7 Actividades de los dos sectores principales
act = df[df["sector"].isin(SECTORES_PRINCIPALES)].groupby(["sector", "actividad"])[V].sum()
verificar("vw_actividades_sector_principal", pd.DataFrame({"total_tco2e": act.round(2)}), ["sector", "actividad"])

OK  vw_balance_anual: 33 filas, diferencia máxima 0.0000


OK  vw_totales_sector: 4 filas, diferencia máxima 0.0000
OK  vw_evolucion_sector_anio: 132 filas, diferencia máxima 0.0000
OK  vw_variacion_sector: 132 filas, diferencia máxima 0.0000
OK  vw_composicion_gas_anio: 158 filas, diferencia máxima 0.0000
OK  vw_top_categorias_recientes: 10 filas, diferencia máxima 0.0000
OK  vw_actividades_sector_principal: 5 filas, diferencia máxima 0.0000


Las siete vistas coinciden con el cálculo independiente en pandas. Con esto, lo que llega a Power BI ya está verificado.

## 6. Exportación a CSV para Power BI

Exportamos la tabla de hechos, las dos dimensiones y las siete vistas a CSV. Son los archivos que se importan en Power BI (Obtener datos > Texto/CSV), donde se arman las relaciones entre `fact_emisiones` y las dimensiones y se construyen las medidas DAX que dependen de los filtros. Mantenemos el formato que ya usa el dashboard.

In [17]:
tablas_y_vistas = [
    "fact_emisiones", "dim_sector", "dim_gas",
    "vw_balance_anual", "vw_totales_sector", "vw_evolucion_sector_anio",
    "vw_variacion_sector", "vw_composicion_gas_anio",
    "vw_top_categorias_recientes", "vw_actividades_sector_principal",
]

for nombre in tablas_y_vistas:
    tabla = pd.read_sql(f"SELECT * FROM {nombre}", conn)
    tabla.to_csv(INTERIM / f"{nombre}.csv", index=False, decimal=",")
    print(f"{nombre}: {len(tabla)} filas --> {(INTERIM / f'{nombre}.csv').relative_to(ROOT)}")

conn.close()

fact_emisiones: 7737 filas --> data\interim\fact_emisiones.csv
dim_sector: 4 filas --> data\interim\dim_sector.csv
dim_gas: 5 filas --> data\interim\dim_gas.csv
vw_balance_anual: 33 filas --> data\interim\vw_balance_anual.csv
vw_totales_sector: 4 filas --> data\interim\vw_totales_sector.csv
vw_evolucion_sector_anio: 132 filas --> data\interim\vw_evolucion_sector_anio.csv
vw_variacion_sector: 132 filas --> data\interim\vw_variacion_sector.csv
vw_composicion_gas_anio: 158 filas --> data\interim\vw_composicion_gas_anio.csv
vw_top_categorias_recientes: 10 filas --> data\interim\vw_top_categorias_recientes.csv
vw_actividades_sector_principal: 5 filas --> data\interim\vw_actividades_sector_principal.csv


## 7. Conclusiones

* El dataset limpio quedó modelado en un esquema en estrella (`fact_emisiones`, `dim_sector` y `dim_gas`) sobre SQLite, sin necesidad de un servidor.
* Las siete vistas resuelven las preguntas del proyecto con CTEs y funciones de ventana (`LAG`, `RANK`, `SUM() OVER`) en lugar de subconsultas anidadas.
* Cada vista se contrastó con un cálculo independiente en pandas, y la tabla de hechos conserva el total del notebook 01 (12.398,33 MtCO₂e).
* **Próximo paso:** importar las tablas y vistas a Power BI, armar las relaciones del modelo y construir las medidas DAX que responden a los filtros.

**Para tener en cuenta:** los nombres de columna conservan `tco2e` aunque los valores estén en MtCO₂e (ver notebook 01, sección 2.4), para no romper el modelo del dashboard.